# 01 · Download GoldenCheetah OpenData (task 1.1)

Data: Liversedge, M. (2020) *GoldenCheetah OpenData Project*, OSF, doi:10.17605/OSF.IO/6HFPZ.
One zip per athlete, saved to Drive under `thesis-cycling/data/raw/goldencheetah/zips/`.

**Everything here is resumable.** If Colab disconnects or a cell fails, run the two setup cells and
then the step again: finished files are skipped.

1. **List** the files on OSF into `manifest.csv` (skipped if it already exists).
2. **Download** in the background. The log goes to a file on Drive, so the page stays light.
   Watch progress with the monitor cell; stopping the monitor does *not* stop the download.
3. **Index** all zips (also resumable), which the audit notebook (02) reads.

*Optional:* if the log shows many "rate limited (429)" pauses, make a free OSF account, create a
personal access token (Settings → Personal access tokens, scope `osf.full_read`) and add it as a Colab
secret named `OSF_TOKEN`.

*Tip:* free Colab stops after a few idle hours or when the tab is closed. For an overnight run, use
your laptop with the same commands (shown in the pull request).

In [ ]:
# --- settings -------------------------------------------------------------
REPO = "Kevin-ai-jpg/BachelorThesis-MachineLearning-RoadCycling"
BRANCH = "main"
DRIVE_DATA = "/content/drive/MyDrive/thesis-cycling/data"
EXTRAS = "track"

In [ ]:
# Same setup as 00_colab_setup: Drive, repo, install, import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess, pathlib, importlib
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)
os.environ["CYCLINGML_DATA"] = DRIVE_DATA
os.environ["CYCLINGML_MLRUNS"] = f"{DRIVE_DATA}/mlruns"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = "/content/repo"
if os.path.isdir(f"{repo_dir}/.git"):
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, repo_dir], check=True)
%cd /content/repo
!git log -1 --oneline
!pip install -q -e ".[{EXTRAS}]"
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
importlib.invalidate_caches()
from cyclingml import paths
GC = paths.raw_dir("goldencheetah")
GC.mkdir(parents=True, exist_ok=True)
print("GoldenCheetah folder:", GC)

In [ ]:
try:
    from google.colab import userdata
    os.environ["OSF_TOKEN"] = userdata.get("OSF_TOKEN")
    print("Using OSF token")
except Exception:
    print("No OSF token, anonymous access")

## 1 · List files on OSF

In [ ]:
from cyclingml.io.osf import read_manifest, summarize
MANIFEST = GC / "manifest.csv"
if not MANIFEST.exists():
    !python -m cyclingml.io.osf list --node 6hfpz --out "{MANIFEST}"
print(summarize(read_manifest(MANIFEST)))

## 2 · Download (runs in the background)

In [ ]:
import subprocess, time
LOG = GC / "download.log"
ZIPS = GC / "zips"
cmd = (f'python -m cyclingml.io.osf download --manifest "{MANIFEST}" --dest "{ZIPS}" '
       f'--workers 3 >> "{LOG}" 2>&1')
with open(LOG, "a") as f:
    f.write(f"\n===== started {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
download = subprocess.Popen(cmd, shell=True, cwd="/content/repo")
print("Download started in the background. Run the next cell to watch it.")

In [ ]:
# Monitor: refreshes every 60 s. Stop this cell any time; the download keeps going.
from IPython.display import clear_output
from cyclingml.io.osf import read_manifest
total = sum(1 for f in read_manifest(MANIFEST) if f.name.endswith(".zip"))
download = globals().get("download")  # None if the session restarted since step 2
while True:
    done = len(list(ZIPS.glob("*.zip")))
    failed = (ZIPS / "failed.txt").read_text().count("\n") if (ZIPS / "failed.txt").exists() else 0
    tail = LOG.read_text().splitlines()[-6:] if LOG.exists() else []
    clear_output(wait=True)
    print(f"{done}/{total} athletes on Drive, {failed} failed this run")
    print("\n".join(tail))
    if download is None:
        print("\n(No download running in this session; start step 2 to resume.)")
        break
    if download.poll() is not None:
        print(f"\nDownload finished (exit code {download.returncode}).",
              "Run step 2 again to retry failures." if failed else "")
        break
    time.sleep(60)

## 3 · Index all zips (resumable)

In [ ]:
!python -m cyclingml.io.goldencheetah index --zips "{ZIPS}" --out "{GC}/index" --workers 2 2>&1 | tail -n 20

Then open `02_data_audit.ipynb`. If something fails, send Claude the last lines of `download.log` (or the cell's error).